In [0]:
%sql
-- ============================================================
-- RETAILBANK EDW MODERNIZATION | PHASE 1: SCHEMA MIGRATION
-- Notebook: 01_Schema_Migration
-- Target:   Unity Catalog | Catalog = retailbank_dev
-- Date:     11 August 2026
-- ============================================================

-- Switch to our catalog so we don't have to prefix every table name.
USE CATALOG retailbank_dev;

-- Create the 6 logical schemas (same layers as SQL Server).
CREATE SCHEMA IF NOT EXISTS source;
CREATE SCHEMA IF NOT EXISTS reference;
CREATE SCHEMA IF NOT EXISTS config;
CREATE SCHEMA IF NOT EXISTS warehouse;
CREATE SCHEMA IF NOT EXISTS reporting;
CREATE SCHEMA IF NOT EXISTS audit;

-- Quick sanity check: we should see 6 schemas plus the default 'information_schema'.
SHOW SCHEMAS IN retailbank_dev;

databaseName
audit
baseline
config
default
information_schema
reference
reporting
source
warehouse


In [0]:
%sql
-- ============================================================
-- SECTION 2: SOURCE TABLES
-- These are the "inboxes" where each operational system drops
-- its daily file. Think of them as raw, unvalidated data.
-- We use STRING instead of VARCHAR because Spark/Delta treats
-- them the same, and STRING is the idiomatic choice.
-- ============================================================
USE CATALOG retailbank_dev;

-- 2.1 Core Banking (CustomerHub / DepositPro)
-- Main current account and savings data.
CREATE OR REPLACE TABLE source.core_banking_accounts (
    account_number   STRING  NOT NULL,  -- Unique account ID in source system
    customer_number  STRING,             -- Links to the customer
    product_code     STRING,             -- e.g. SAV001, CUR001
    account_balance  DECIMAL(18,2),      -- Money in the account
    currency         STRING,             -- USD, EUR, GBP, etc.
    account_status   STRING,             -- ACTIVE or CLOSED
    record_date      DATE,               -- When this row was sent
    CONSTRAINT pk_source_core_banking_accounts PRIMARY KEY (account_number)
);

-- 2.2 Loans (LoanSphere)
-- Mortgage and personal loan data.
CREATE OR REPLACE TABLE source.loan_accounts (
    loan_number        STRING  NOT NULL,
    client_id          STRING,
    loan_product       STRING,           -- e.g. HOME_LOAN
    outstanding_amount DECIMAL(18,2),    -- How much is still owed
    loan_currency      STRING,
    loan_status        STRING,
    snapshot_date      DATE,
    CONSTRAINT pk_source_loan_accounts PRIMARY KEY (loan_number)
);

-- 2.3 Cards (CardMaster)
-- Credit card account data.
CREATE OR REPLACE TABLE source.card_accounts (
    card_account_number STRING  NOT NULL,
    client_number       STRING,
    card_product        STRING,          -- e.g. GOLD_CARD
    credit_balance      DECIMAL(18,2),   -- Current balance
    card_currency       STRING,
    card_status         STRING,
    business_date       DATE,
    CONSTRAINT pk_source_card_accounts PRIMARY KEY (card_account_number)
);

-- 2.4 Investments (WealthPlus)
-- Investment fund and portfolio holdings.
CREATE OR REPLACE TABLE source.investment_accounts (
    investment_id       STRING  NOT NULL,
    investor_id         STRING,
    investment_product  STRING,          -- e.g. EQUITY_FUND
    market_value        DECIMAL(18,2),   -- Current value
    investment_currency STRING,
    investment_status   STRING,
    valuation_date      DATE,
    CONSTRAINT pk_source_investment_accounts PRIMARY KEY (investment_id)
);

-- 2.5 Mortgage
-- Separate mortgage account data.
CREATE OR REPLACE TABLE source.mortgage_accounts (
    mortgage_number STRING  NOT NULL,
    client_code     STRING,
    mortgage_type   STRING,              -- e.g. HOME
    balance_amount  DECIMAL(18,2),
    currency_code   STRING,
    status          STRING,
    effective_date  DATE,
    CONSTRAINT pk_source_mortgage_accounts PRIMARY KEY (mortgage_number)
);

-- 2.6 Mobile Banking Wallet
-- Digital wallet balances.
CREATE OR REPLACE TABLE source.mobile_wallet (
    wallet_id       STRING  NOT NULL,
    customer_number STRING,
    wallet_product  STRING,              -- e.g. MOBILE_WALLET
    wallet_balance  DECIMAL(18,2),
    wallet_currency STRING,
    wallet_status   STRING,
    load_date       DATE,
    CONSTRAINT pk_source_mobile_wallet PRIMARY KEY (wallet_id)
);

-- 2.7 Forex (FXConnect)
-- Foreign exchange trading accounts.
CREATE OR REPLACE TABLE source.forex_accounts (
    trade_id      STRING  NOT NULL,
    customer_id   STRING,
    forex_product STRING,                -- e.g. FX_FORWARD
    trade_amount  DECIMAL(18,2),
    trade_currency STRING,
    trade_status  STRING,
    trade_date    DATE,
    CONSTRAINT pk_source_forex_accounts PRIMARY KEY (trade_id)
);

In [0]:
%sql
-- ============================================================
-- SECTION 3: REFERENCE TABLES
-- These are like dictionaries. They don't change every day.
-- They provide the "meanings" for codes used in source data.
-- Example: ProductCode = 'SAV001' means 'Savings Account'.
-- ============================================================
USE CATALOG retailbank_dev;

-- 3.1 Product Reference
-- Maps product codes to human-readable names and categories.
CREATE OR REPLACE TABLE reference.product (
    product_code        STRING  NOT NULL,
    product_description STRING,
    product_category    STRING,
    regulatory_category STRING,
    product_status      STRING,
    effective_date      DATE,
    CONSTRAINT pk_reference_product PRIMARY KEY (product_code)
);

-- 3.2 Exchange Rates
-- Converts foreign currencies to USD (the base currency).
-- We need this because accounts are in EUR/GBP but reports are in USD.
-- The PK is composite: (CurrencyCode, EffectiveDate).
CREATE OR REPLACE TABLE reference.exchange_rates (
    currency_code  STRING  NOT NULL,
    exchange_rate  DECIMAL(18,8),
    effective_date DATE    NOT NULL,
    CONSTRAINT pk_reference_exchange_rates PRIMARY KEY (currency_code, effective_date)
);

In [0]:
%sql
-- ============================================================
-- SECTION 4: CONFIGURATION TABLES
-- These are the "control panel" of the data warehouse.
-- Business users can change rules here without rewriting code.
--
-- We split Config.SourceConfiguration into TWO tables because
-- Customer loading and Portfolio loading need different columns.
-- Trying to cram both into one table caused inconsistency #6.
-- ============================================================
USE CATALOG retailbank_dev;

-- 4.1 Customer Source Configuration
-- Tells Procedure 1 WHERE to find customer data and in WHAT ORDER.
CREATE OR REPLACE TABLE config.customer_source_configuration (
    source_system_id     INT     NOT NULL,
    source_system_code   STRING,
    source_system_name   STRING,
    customer_table       STRING,   -- e.g. source.core_banking_accounts
    is_active            BOOLEAN,  -- true = use this source, false = skip
    load_priority        INT,      -- 1 = first, 7 = last
    supports_incremental BOOLEAN,  -- Can we load only changed rows?
    CONSTRAINT pk_config_customer_source_config PRIMARY KEY (source_system_id)
);

-- 4.2 Portfolio Source Configuration
-- Tells Procedure 3 WHERE to find product/account data.
-- Also maps COLUMN NAMES because each system uses different names.
CREATE OR REPLACE TABLE config.portfolio_source_configuration (
    source_system_id     INT     NOT NULL,
    source_system_code   STRING,
    source_system_name   STRING,
    product_table        STRING,   -- Which table has the data
    customer_field       STRING,   -- Column name for customer ID
    product_field        STRING,   -- Column name for product code
    balance_field        STRING,   -- Column name for balance
    currency_field       STRING,   -- Column name for currency
    status_field         STRING,   -- Column name for account status
    business_date_field  STRING,   -- Column name for the date
    source_account_field STRING,   -- Column name for account number
    load_priority        INT,
    supports_incremental BOOLEAN,
    is_active            BOOLEAN,
    CONSTRAINT pk_config_portfolio_source_config PRIMARY KEY (source_system_id)
);

-- 4.3 Exception Rules
-- The 6 Data Quality (DQ) rules that Procedure 4 runs every night.
CREATE OR REPLACE TABLE config.exception_rules (
    rule_code          STRING  NOT NULL,
    exception_category STRING,
    severity_code      STRING,
    business_area      STRING,
    rule_enabled       BOOLEAN,
    CONSTRAINT pk_config_exception_rules PRIMARY KEY (rule_code)
);

-- 4.4 Exception Category
-- Groups exceptions into categories and assigns default severity.
CREATE OR REPLACE TABLE config.exception_category (
    exception_category STRING  NOT NULL,
    description        STRING,
    default_severity   STRING,
    business_area      STRING,
    CONSTRAINT pk_config_exception_category PRIMARY KEY (exception_category)
);

-- 4.5 Exception Severity
-- Defines what each severity level means in business terms.
CREATE OR REPLACE TABLE config.exception_severity (
    severity_code        STRING  NOT NULL,
    priority_level       INT,     -- 1 = most urgent, 4 = least
    sla_hours            INT,     -- How many hours to fix it
    escalation_required  BOOLEAN, -- true = tell the boss
    CONSTRAINT pk_config_exception_severity PRIMARY KEY (severity_code)
);

-- 4.6 Business Owner
-- Maps each business area to the team that handles exceptions.
CREATE OR REPLACE TABLE config.business_owner (
    business_area  STRING,
    business_owner STRING,
    support_team   STRING
);

-- 4.7 Product Category
-- Groups products into regulatory categories.
CREATE OR REPLACE TABLE config.product_category (
    product_category    STRING,
    regulatory_category STRING,
    active_flag         BOOLEAN
);

-- 4.8 Portfolio Value Bands
-- Defines customer segments based on total portfolio value.
CREATE OR REPLACE TABLE config.portfolio_value_band (
    band_name    STRING,
    minimum_value DECIMAL(18,2),
    maximum_value DECIMAL(18,2)
);

-- 4.9 Reporting Thresholds
-- Defines RED/AMBER/GREEN thresholds for executive dashboards.
CREATE OR REPLACE TABLE config.reporting_threshold (
    metric_name     STRING,
    green_threshold DECIMAL(18,2),
    amber_threshold DECIMAL(18,2),
    red_threshold   DECIMAL(18,2)
);

-- 4.10 Source System Lookup
-- Simple list of all source systems for dropdown menus and reports.
CREATE OR REPLACE TABLE config.source_system (
    source_system_code STRING,
    description        STRING,
    active_flag        BOOLEAN
);

In [0]:
%sql
-- ============================================================
-- SECTION 5: WAREHOUSE TABLES
-- These hold the CLEAN, TRUSTED data -- the "single source of truth".
-- This is what business users and reports actually look at.
--
-- We use GENERATED ALWAYS AS IDENTITY instead of IDENTITY(1,1).
-- Delta Lake manages this automatically. We never insert into
-- these identity columns -- the engine does it for us.
-- ============================================================
USE CATALOG retailbank_dev;

-- 5.1 Customer Master
-- The "Golden Record": one row per customer, cleaned and deduplicated.
CREATE OR REPLACE TABLE warehouse.customer_master (
    customer_id       STRING  NOT NULL,
    first_name        STRING,
    last_name         STRING,
    customer_category STRING,
    branch_code       STRING,
    customer_status   STRING,
    created_date      TIMESTAMP,  -- Set explicitly by nb_01
    last_updated_date TIMESTAMP,  -- Set explicitly by nb_01
    CONSTRAINT pk_warehouse_customer_master PRIMARY KEY (customer_id)
);

-- 5.2 Customer Master Exceptions
-- Records data quality problems found during customer loading.
-- Expanded from 5 columns to 20 to match what Procedure 2 expects.
CREATE OR REPLACE TABLE warehouse.customer_master_exceptions (
    exception_id           BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date          DATE,
    customer_id            STRING,
    source_system_code     STRING,
    exception_category     STRING,
    exception_description  STRING,
    severity_code          STRING,
    severity_description   STRING,
    escalation_required    BOOLEAN,
    resolution_target_hours INT,
    business_area          STRING,
    business_owner         STRING,
    support_team           STRING,
    logged_date            TIMESTAMP,
    exception_status       STRING,
    assigned_date          TIMESTAMP,
    resolution_status      STRING,
    escalation_level       INT,
    resolution_due_date    TIMESTAMP,
    created_date           TIMESTAMP,
    last_updated_date      TIMESTAMP,
    CONSTRAINT pk_warehouse_customer_master_exc PRIMARY KEY (exception_id)
);

-- 5.3 Customer Portfolio
-- The complete view of everything a customer owns across all 7 systems.
-- This is one of the main project goals: "single portfolio view".
CREATE OR REPLACE TABLE warehouse.customer_portfolio (
    portfolio_id            BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date           DATE    NOT NULL,
    source_system_code      STRING,
    source_account_number   STRING  NOT NULL,
    customer_id             STRING,
    customer_name           STRING,
    customer_category       STRING,
    branch_code             STRING,
    product_code            STRING,
    product_description     STRING,
    product_category        STRING,
    regulatory_category     STRING,
    currency_code           STRING,
    exchange_rate           DECIMAL(18,8),
    account_balance         DECIMAL(18,2),
    base_currency_balance   DECIMAL(18,2),
    account_status          STRING,
    eligible_for_reporting  STRING,
    portfolio_value_band    STRING,
    high_value_customer     BOOLEAN,
    product_count           INT,
    created_date            TIMESTAMP,
    last_updated_date       TIMESTAMP,
    CONSTRAINT pk_warehouse_customer_portfolio PRIMARY KEY (portfolio_id)
);

-- 5.4 Customer Portfolio Exceptions
-- Data quality problems found in portfolio data.
-- Enriched with hashes, SLAs, escalation queues, etc.
CREATE OR REPLACE TABLE warehouse.customer_portfolio_exceptions (
    exception_id          BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date         DATE,
    source_system_code    STRING,
    source_account_number STRING,
    customer_id           STRING,
    rule_code             STRING,
    exception_category    STRING,
    exception_description STRING,
    exception_value       STRING,
    severity_code         STRING,
    priority_level        INT,
    sla_hours             INT,
    escalation_required   BOOLEAN,
    escalation_queue      STRING,
    business_area         STRING,
    business_owner        STRING,
    exception_hash        STRING,
    logged_date           TIMESTAMP,
    resolution_due_date   TIMESTAMP,
    created_date          TIMESTAMP,
    last_updated_date     TIMESTAMP,
    CONSTRAINT pk_warehouse_customer_portfolio_exc PRIMARY KEY (exception_id)
);

-- 6.1 Portfolio Errors (NEW -- Inconsistency #8 RESOLVED)
-- A simple staging table for raw validation errors from Procedure 3.
-- The old code tried to insert 7 columns into a 20+ column table and crashed.
-- This table has exactly the 7 columns Procedure 3 produces.
CREATE OR REPLACE TABLE warehouse.portfolio_errors (
    error_id              BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date         DATE,
    source_system_code    STRING,
    source_account_number STRING,
    customer_id           STRING,
    error_category        STRING,
    error_description     STRING,
    logged_date           TIMESTAMP,
    execution_id          STRING,
    CONSTRAINT pk_warehouse_portfolio_errors PRIMARY KEY (error_id)
);

In [0]:
%sql
-- ============================================================
-- SECTION 6: AUDIT TABLES
-- The "black box recorder" of the data warehouse.
-- Answers: Did last night's batch run? How many rows? Any errors?
-- ============================================================
USE CATALOG retailbank_dev;

-- 6.1 ETL Execution Log (Master)
-- Every procedure writes one row here when it starts, updates when done.
CREATE OR REPLACE TABLE audit.etl_execution_log (
    execution_id     STRING  NOT NULL,
    procedure_name   STRING  NOT NULL,
    business_date    DATE,
    load_type        STRING,
    start_time       TIMESTAMP,
    end_time         TIMESTAMP,
    status           STRING,     -- RUNNING, SUCCESS, or FAILED
    rows_read        INT,
    rows_inserted    INT,
    rows_updated     INT,
    rows_rejected    INT,
    duration_seconds INT,
    message          STRING,
    created_date     TIMESTAMP,
    CONSTRAINT pk_audit_etl_execution_log PRIMARY KEY (execution_id)
);

-- 6.2 Data Quality Issues (NEW -- Inconsistency #4 RESOLVED)
-- Procedures 1 and 2 referenced this table, but it never existed.
CREATE OR REPLACE TABLE audit.data_quality_issues (
    issue_id          BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date     DATE    NOT NULL,
    source_system     STRING  NOT NULL,
    customer_id       STRING,
    error_category    STRING  NOT NULL,
    error_description STRING  NOT NULL,
    logged_date       TIMESTAMP,
    execution_id      STRING,
    CONSTRAINT pk_audit_data_quality_issues PRIMARY KEY (issue_id)
);

-- 6.3 Exception Execution Summary (NEW -- Inconsistency #5 RESOLVED)
-- Separate audit table for customer exceptions (Procedure 2).
CREATE OR REPLACE TABLE audit.exception_execution_summary (
    summary_id            BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date         DATE,
    execution_id          STRING  NOT NULL,
    procedure_name        STRING,
    total_exceptions      INT,
    critical_exceptions   INT,
    high_exceptions       INT,
    medium_exceptions     INT,
    low_exceptions        INT,
    escalated_exceptions  INT,
    execution_seconds     INT,
    created_date          TIMESTAMP,
    CONSTRAINT pk_audit_exception_execution_summ PRIMARY KEY (summary_id)
);

-- 6.4 Portfolio Execution Summary
-- Daily statistics from Procedure 3 (portfolio loading).
CREATE OR REPLACE TABLE audit.portfolio_execution_summary (
    summary_id             BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date          DATE,
    procedure_name         STRING,
    execution_id           STRING,
    total_customers        INT,
    total_accounts         INT,
    total_portfolio_value  DECIMAL(18,2),
    total_errors           INT,
    execution_time_seconds INT,
    created_date           TIMESTAMP,
    CONSTRAINT pk_audit_portfolio_execution_summ PRIMARY KEY (summary_id)
);

-- 6.5 Portfolio Exception Execution Summary
-- Daily statistics from Procedure 4 (exception processing).
CREATE OR REPLACE TABLE audit.portfolio_exception_execution_summary (
    summary_id            BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date         DATE,
    execution_id          STRING,
    procedure_name        STRING,
    total_exceptions      INT,
    critical_exceptions   INT,
    high_exceptions       INT,
    medium_exceptions     INT,
    low_exceptions        INT,
    escalated_exceptions  INT,
    execution_seconds     INT,
    created_date          TIMESTAMP,
    CONSTRAINT pk_audit_portfolio_exception_exec_summ PRIMARY KEY (summary_id)
);

-- 6.6 Business Area Exception Summary
-- Shows which departments have the most problems.
CREATE OR REPLACE TABLE audit.business_area_exception_summary (
    summary_id      BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date   DATE,
    business_area   STRING,
    exception_count INT,
    critical_count  INT,
    created_date    TIMESTAMP,
    CONSTRAINT pk_audit_business_area_exception_summ PRIMARY KEY (summary_id)
);

-- 6.7 Reporting Execution Summary
-- Tracks which reports were generated and how long they took.
CREATE OR REPLACE TABLE audit.reporting_execution_summary (
    summary_id        BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date     DATE,
    execution_id      STRING,
    procedure_name    STRING,
    report_name       STRING,
    records_generated INT,
    execution_seconds INT,
    created_date      TIMESTAMP,
    CONSTRAINT pk_audit_reporting_execution_summ PRIMARY KEY (summary_id)
);

-- 6.8 Executive Dashboard Metrics
-- Snapshot of portfolio health for the executive dashboard.
CREATE OR REPLACE TABLE audit.executive_dashboard_metrics (
    metric_id            BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date        DATE,
    portfolio_health     STRING,
    critical_exceptions  INT,
    exception_rate       DECIMAL(10,2),
    sla_compliance       DECIMAL(10,2),
    generated_date       TIMESTAMP,
    CONSTRAINT pk_audit_executive_dashboard_metrics PRIMARY KEY (metric_id)
);

-- 6.9 Data Lineage (Recommended)
-- Tracks where each piece of data came from and how it was transformed.
CREATE OR REPLACE TABLE audit.data_lineage (
    lineage_id           BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    procedure_name       STRING,
    source_object        STRING,
    source_column        STRING,
    transformation_rule  STRING,
    target_object        STRING,
    target_column        STRING,
    business_rule        STRING,
    created_date         TIMESTAMP,
    CONSTRAINT pk_audit_data_lineage PRIMARY KEY (lineage_id)
);

-- 6.10 ETL Error Log (Recommended)
-- Captures detailed error messages when a procedure fails.
CREATE OR REPLACE TABLE audit.etl_error_log (
    error_id       BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    execution_id   STRING,
    procedure_name STRING,
    error_number   INT,
    error_message  STRING,
    error_date     TIMESTAMP,
    CONSTRAINT pk_audit_etl_error_log PRIMARY KEY (error_id)
);

In [0]:
%sql
-- ============================================================
-- SECTION 7: REPORTING TABLES
-- These feed the executive dashboards and Power BI reports.
-- Procedure 5 TRUNCATEs and reloads this table every night.
-- ============================================================
USE CATALOG retailbank_dev;

CREATE OR REPLACE TABLE reporting.portfolio_exception_report (
    report_id    BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    business_date DATE,
    kpi_name      STRING,   -- e.g. ExceptionRate, PortfolioHealth
    kpi_value     STRING,   -- The actual value
    kpi_status    STRING,   -- GREEN, AMBER, RED
    created_date  TIMESTAMP,
    CONSTRAINT pk_reporting_portfolio_exception_rpt PRIMARY KEY (report_id)
);

In [0]:
%sql
-- ============================================================
-- SECTION 8: CONFIGURATION DATA
-- These are the "settings" that control how the system behaves.
-- IMPORTANT: These values have been RECONCILED across all files.
-- They represent the single agreed-upon version of the truth.
-- ============================================================
USE CATALOG retailbank_dev;

-- 8.1 Customer Source Configuration
INSERT INTO config.customer_source_configuration
VALUES
(1, 'CORE_BANKING', 'Core Banking System', 'source.core_banking_accounts', true, 1, true),
(2, 'LOANS',        'Loan System',         'source.loan_accounts',        true, 2, true),
(3, 'CARDS',        'Card Platform',       'source.card_accounts',        true, 3, true),
(4, 'INVESTMENTS',  'Investment Platform', 'source.investment_accounts',  true, 4, true),
(5, 'MORTGAGE',     'Mortgage System',     'source.mortgage_accounts',    true, 5, true),
(6, 'MOBILE',       'Mobile Banking',      'source.mobile_wallet',        true, 6, true),
(7, 'FOREX',        'Forex Platform',      'source.forex_accounts',       true, 7, true);

-- 8.2 Portfolio Source Configuration
INSERT INTO config.portfolio_source_configuration
VALUES
(1, 'CORE_BANKING', 'Core Banking System', 'source.core_banking_accounts', 'customer_number', 'product_code', 'account_balance', 'currency', 'account_status', 'record_date', 'account_number', 1, true, true),
(2, 'LOANS',        'Loan System',         'source.loan_accounts',        'client_id',       'loan_product', 'outstanding_amount', 'loan_currency', 'loan_status', 'snapshot_date', 'loan_number', 2, true, true),
(3, 'CARDS',        'Card Platform',       'source.card_accounts',        'client_number',   'card_product', 'credit_balance', 'card_currency', 'card_status', 'business_date', 'card_account_number', 3, true, true),
(4, 'INVESTMENTS',  'Investment Platform', 'source.investment_accounts',  'investor_id',     'investment_product', 'market_value', 'investment_currency', 'investment_status', 'valuation_date', 'investment_id', 4, true, true),
(5, 'MORTGAGE',     'Mortgage System',     'source.mortgage_accounts',    'client_code',     'mortgage_type', 'balance_amount', 'currency_code', 'status', 'effective_date', 'mortgage_number', 5, true, true),
(6, 'MOBILE',       'Mobile Banking',      'source.mobile_wallet',        'customer_number', 'wallet_product', 'wallet_balance', 'wallet_currency', 'wallet_status', 'load_date', 'wallet_id', 6, true, true),
(7, 'FOREX',        'Forex Platform',      'source.forex_accounts',       'customer_id',     'forex_product', 'trade_amount', 'trade_currency', 'trade_status', 'trade_date', 'trade_id', 7, true, true);

-- 8.3 Exception Rules (Authoritative -- Inconsistency #10 RESOLVED)
INSERT INTO config.exception_rules
VALUES
('DQ001', 'Customer Validation', 'HIGH',     'Customer Operations', true),
('DQ002', 'Product Validation',  'MEDIUM',   'Product Management',  true),
('DQ003', 'Currency Validation', 'HIGH',     'Finance',             true),
('DQ004', 'Financial Validation', 'CRITICAL', 'Risk',                true),
('DQ005', 'Business Rule',       'HIGH',     'Operations',          true),
('DQ006', 'Duplicate Account',   'CRITICAL', 'Data Governance',     true);

-- 8.4 Exception Category
INSERT INTO config.exception_category
VALUES
('Customer Validation', 'Customer master validation',           'HIGH',     'Customer Operations'),
('Product Validation',  'Product reference validation',         'MEDIUM',   'Product Management'),
('Currency Validation', 'Exchange rate validation',             'HIGH',     'Finance'),
('Financial Validation', 'Portfolio financial checks',          'CRITICAL', 'Risk'),
('Business Rule',       'Business processing rules',            'HIGH',     'Operations'),
('Duplicate Account',   'Duplicate portfolio account detection', 'CRITICAL', 'Data Governance');

-- 8.5 Exception Severity
INSERT INTO config.exception_severity
VALUES
('CRITICAL', 1, 2,  true),
('HIGH',     2, 8,  true),
('MEDIUM',   3, 24, false),
('LOW',      4, 72, false);

-- 8.6 Business Owner (Inconsistency #11 RESOLVED)
INSERT INTO config.business_owner
VALUES
('Customer Operations', 'Customer Services Team', 'CRM Support'),
('Product Management',  'Product Owners',         'Product Support'),
('Finance',             'Finance Control',        'Finance Operations'),
('Risk',                'Risk Management',        'Risk Analytics'),
('Operations',          'Operations Team',        'Operations Support'),
('Data Governance',     'Data Quality Office',    'Enterprise Data Office');

-- 8.7 Product Category
INSERT INTO config.product_category
VALUES
('Deposit',    'Retail',     true),
('Loan',       'Credit',     true),
('Mortgage',   'Credit',     true),
('Investment', 'Investment', true),
('Card',       'Retail',     true),
('Forex',      'Treasury',   true);

-- 8.8 Portfolio Value Bands
INSERT INTO config.portfolio_value_band
VALUES
('STANDARD', 0.00,       49999.99),
('SILVER',   50000.00,   249999.99),
('GOLD',     250000.00,  999999.99),
('PLATINUM', 1000000.00, 999999999.99);

-- 8.9 Reporting Thresholds (Inconsistency #12 RESOLVED)
INSERT INTO config.reporting_threshold
VALUES
('Exception Rate',      2.00,  5.00,   100.00),
('Critical Exceptions', 10.00, 50.00,  100000.00),
('SLA Failure Rate',    5.00,  15.00,  100.00);

-- 8.10 Source System Lookup
INSERT INTO config.source_system
VALUES
('CORE_BANKING', 'Core Banking',      true),
('LOANS',        'Loan System',       true),
('CARDS',        'Card Processing',   true),
('INVESTMENTS',  'Investment Platform', true),
('MORTGAGE',     'Mortgage Platform', true),
('FOREX',        'Foreign Exchange',  true),
('MOBILE',       'Mobile Banking',    true);

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
-- ============================================================
-- SECTION 9: REFERENCE DATA
-- Product catalog and exchange rates.
-- These are the "dictionaries" that give meaning to source codes.
-- ============================================================
USE CATALOG retailbank_dev;

INSERT INTO reference.product
VALUES
('SAV001',        'Savings Account',                'Deposit',      'Retail',     'ACTIVE', '2026-01-01'),
('CUR001',        'Current Account',                'Deposit',      'Retail',     'ACTIVE', '2026-01-01'),
('HOME_LOAN',     'Residential Mortgage Loan',      'Loans',        'Credit',     'ACTIVE', '2026-01-01'),
('PERSONAL_LOAN', 'Personal Lending Product',       'Loans',        'Credit',     'ACTIVE', '2026-01-01'),
('GOLD_CARD',     'Premium Credit Card',            'Cards',        'Credit',     'ACTIVE', '2026-01-01'),
('STANDARD_CARD', 'Standard Credit Card',           'Cards',        'Credit',     'ACTIVE', '2026-01-01'),
('EQUITY_FUND',   'Equity Investment Fund',         'Investments',  'Investment', 'ACTIVE', '2026-01-01'),
('BOND_FUND',     'Fixed Income Investment Fund',   'Investments',  'Investment', 'ACTIVE', '2026-01-01'),
('MOBILE_WALLET', 'Digital Wallet',                 'Digital',      'Payment',    'ACTIVE', '2026-01-01'),
('FX_FORWARD',    'Foreign Exchange Forward',       'Forex',        'Treasury',   'ACTIVE', '2026-01-01');

INSERT INTO reference.exchange_rates
VALUES
('USD', 1.00000000, '2026-01-31'),
('EUR', 1.08000000, '2026-01-31'),
('GBP', 1.25000000, '2026-01-31'),
('ZAR', 0.05500000, '2026-01-31'),
('JPY', 0.00670000, '2026-01-31');

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
-- ============================================================
-- SECTION 10: DEPLOYMENT VERIFICATION
-- Count tables per schema. Should total 35.
-- ============================================================
USE CATALOG retailbank_dev;

SELECT 'source'    AS layer, COUNT(*) AS table_count FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'source'
UNION ALL
SELECT 'reference', COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'reference'
UNION ALL
SELECT 'config',    COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'config'
UNION ALL
SELECT 'warehouse', COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'warehouse'
UNION ALL
SELECT 'reporting', COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'reporting'
UNION ALL
SELECT 'audit',     COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE table_schema = 'audit';

layer,table_count
source,7
reference,2
config,10
warehouse,5
reporting,1
audit,10


In [0]:
%sql
-- ============================================================
-- SECTION 11: SAMPLE SOURCE DATA (Day 1)
-- This is the raw data that arrives from your 7 operational
-- systems on business date 2026-01-31. It includes intentional
-- data quality issues so we can test the exception rules.
-- ============================================================
USE CATALOG retailbank_dev;

-- 11.1 Core Banking
-- Includes NULL CustomerId (CB10005), NULL ProductCode (CB10006),
-- NULL Currency (CB10007), Negative Balance (CB10004),
-- and CLOSED account (CB10008).
INSERT INTO source.core_banking_accounts
VALUES
('CB10001', 'CUST001', 'SAV001', 25000.00,  'USD', 'ACTIVE', '2026-01-31'),
('CB10002', 'CUST002', 'CUR001', 75000.00,  'EUR', 'ACTIVE', '2026-01-31'),
('CB10003', 'CUST003', 'SAV001', 0.00,      'USD', 'ACTIVE', '2026-01-31'),
('CB10004', 'CUST004', 'CUR001', -5000.00,  'GBP', 'ACTIVE', '2026-01-31'),
('CB10005', NULL,      'SAV001', 10000.00,  'USD', 'ACTIVE', '2026-01-31'),
('CB10006', 'CUST006', NULL,     15000.00,  'USD', 'ACTIVE', '2026-01-31'),
('CB10007', 'CUST007', 'SAV001', 45000.00,  NULL,  'ACTIVE', '2026-01-31'),
('CB10008', 'CUST008', 'SAV001', 8000.00,   'USD', 'CLOSED', '2026-01-31');

-- 11.2 Loans
-- Includes negative balance (LN10004) and unknown product (LN10005).
INSERT INTO source.loan_accounts
VALUES
('LN10001', 'CUST001', 'HOME_LOAN',     500000, 'USD', 'ACTIVE', '2026-01-31'),
('LN10002', 'CUST002', 'PERSONAL_LOAN', 25000,  'EUR', 'ACTIVE', '2026-01-31'),
('LN10003', 'CUST009', 'HOME_LOAN',     750000, 'GBP', 'ACTIVE', '2026-01-31'),
('LN10004', 'CUST010', 'PERSONAL_LOAN', -1000,  'USD', 'ACTIVE', '2026-01-31'),
('LN10005', 'CUST011', 'UNKNOWN',       30000,  'USD', 'ACTIVE', '2026-01-31');

-- 11.3 Cards
-- Includes NULL balance (CARD004).
INSERT INTO source.card_accounts
VALUES
('CARD001', 'CUST001', 'GOLD_CARD',      12000, 'USD', 'ACTIVE', '2026-01-31'),
('CARD002', 'CUST003', 'STANDARD_CARD',  5000,  'USD', 'ACTIVE', '2026-01-31'),
('CARD003', 'CUST004', 'GOLD_CARD',      25000, 'EUR', 'ACTIVE', '2026-01-31'),
('CARD004', 'CUST005', 'STANDARD_CARD',  NULL,  'USD', 'ACTIVE', '2026-01-31');

-- 11.4 Investments
INSERT INTO source.investment_accounts
VALUES
('INV001', 'CUST001', 'EQUITY_FUND', 150000,  'USD', 'ACTIVE', '2026-01-31'),
('INV002', 'CUST012', 'BOND_FUND',   250000,  'EUR', 'ACTIVE', '2026-01-31'),
('INV003', 'CUST013', 'EQUITY_FUND', 1000000, 'USD', 'ACTIVE', '2026-01-31');

-- 11.5 Mortgage
INSERT INTO source.mortgage_accounts
VALUES
('MORT001', 'CUST001', 'HOME', 900000,  'USD', 'ACTIVE', '2026-01-31'),
('MORT002', 'CUST014', 'HOME', 1200000, 'USD', 'ACTIVE', '2026-01-31'),
('MORT003', 'CUST015', 'HOME', 500000,  'GBP', 'CLOSED', '2026-01-31');

-- 11.6 Mobile Wallet
INSERT INTO source.mobile_wallet
VALUES
('WAL001', 'CUST001', 'MOBILE_WALLET', 500,  'USD', 'ACTIVE', '2026-01-31'),
('WAL002', 'CUST020', 'MOBILE_WALLET', 1000, 'EUR', 'ACTIVE', '2026-01-31'),
('WAL003', 'CUST021', 'MOBILE_WALLET', 0,    'USD', 'ACTIVE', '2026-01-31');

-- 11.7 Forex
INSERT INTO source.forex_accounts
VALUES
('FX001', 'CUST001', 'FX_FORWARD', 250000,  'USD', 'ACTIVE', '2026-01-31'),
('FX002', 'CUST030', 'FX_SWAP',    500000,  'EUR', 'ACTIVE', '2026-01-31'),
('FX003', 'CUST031', 'FX_FORWARD', -50000,  'GBP', 'ACTIVE', '2026-01-31');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- ============================================================
-- SECTION 12: SAMPLE CUSTOMER MASTER DATA
-- This simulates the output of Procedure 1 after it has run.
-- In the real pipeline, nb_01 would create these rows from the
-- 7 source tables. For now, we seed them so nb_03 (Portfolio)
-- can join to them during testing.
-- ============================================================
USE CATALOG retailbank_dev;

INSERT INTO warehouse.customer_master
(customer_id, first_name, last_name, customer_category, branch_code, customer_status, created_date, last_updated_date)
VALUES
('CUST001', 'John',    'Smith',    'PREMIUM',  'BR001', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST002', 'Mary',    'Jones',    'STANDARD', 'BR002', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST003', 'Peter',   'Brown',    'STANDARD', 'BR001', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST004', 'Sarah',   'Williams', 'PREMIUM',  'BR003', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST006', 'David',   'Miller',   'BUSINESS', 'BR004', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST007', 'James',   'Wilson',   'STANDARD', 'BR002', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST008', 'Linda',   'Taylor',   'STANDARD', 'BR001', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST010', 'Robert',  'Johnson',  'BUSINESS', 'BR005', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
('CUST012', 'Michael', 'Davis',    'PREMIUM',  'BR006', 'ACTIVE', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
-- ============================================================
-- SECTION 13: DATA LOAD VERIFICATION
-- Count rows in every table that should now have data.
-- ============================================================
USE CATALOG retailbank_dev;

SELECT 'source.core_banking_accounts'  AS table_name, COUNT(*) AS row_count FROM source.core_banking_accounts
UNION ALL SELECT 'source.loan_accounts',           COUNT(*) FROM source.loan_accounts
UNION ALL SELECT 'source.card_accounts',           COUNT(*) FROM source.card_accounts
UNION ALL SELECT 'source.investment_accounts',     COUNT(*) FROM source.investment_accounts
UNION ALL SELECT 'source.mortgage_accounts',       COUNT(*) FROM source.mortgage_accounts
UNION ALL SELECT 'source.mobile_wallet',           COUNT(*) FROM source.mobile_wallet
UNION ALL SELECT 'source.forex_accounts',          COUNT(*) FROM source.forex_accounts
UNION ALL SELECT 'reference.product',              COUNT(*) FROM reference.product
UNION ALL SELECT 'reference.exchange_rates',       COUNT(*) FROM reference.exchange_rates
UNION ALL SELECT 'config.customer_source_configuration', COUNT(*) FROM config.customer_source_configuration
UNION ALL SELECT 'config.portfolio_source_configuration', COUNT(*) FROM config.portfolio_source_configuration
UNION ALL SELECT 'config.exception_rules',         COUNT(*) FROM config.exception_rules
UNION ALL SELECT 'warehouse.customer_master',      COUNT(*) FROM warehouse.customer_master;

table_name,row_count
source.core_banking_accounts,8
source.loan_accounts,5
source.card_accounts,4
source.investment_accounts,3
source.mortgage_accounts,3
source.mobile_wallet,3
source.forex_accounts,3
reference.product,10
reference.exchange_rates,5
config.customer_source_configuration,7
